In [ ]:
import os

# 1. Set the cache directory BEFORE importing TensorFlow/Keras.
# For reliable offline use, replace with a fixed path, e.g. r"D:\ml_project"
BASE_DIR = os.getcwd()
os.environ["KERAS_HOME"] = os.path.join(BASE_DIR, "my_keras_cache")

import numpy as np
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

# =====================================================================
# STAGE A: Loading and Preprocessing the Image Data
# =====================================================================
print("--- Stage A: Loading and preparing the CIFAR-10 dataset ---")
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values from [0, 255] to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


# =====================================================================
# STAGE B: Defining the Model's Architecture
# =====================================================================
print("\n--- Stage B: Defining the CNN model architecture ---")
model = Sequential(
    [
        Input(shape=(32, 32, 3)),
        # Block 1: low-level features (edges)
        Conv2D(32, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),
        # Block 2: mid-level features (shapes)
        Conv2D(64, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),
        # Block 3: high-level features (textures)
        Conv2D(128, (3, 3), activation="relu", padding="same"),
        MaxPooling2D((2, 2)),
        # Classifier
        Flatten(),
        Dense(128, activation="relu"),
        Dense(10, activation="softmax"),
    ]
)

model.summary()

print("\nCompiling the model...")
model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)


# =====================================================================
# STAGE C: Training the Model
# =====================================================================
print("\n--- Stage C: Training the model ---")
history = model.fit(
    x_train, y_train, epochs=15, batch_size=64, validation_data=(x_test, y_test)
)


# =====================================================================
# STAGE D: Estimating the Model's Performance
# =====================================================================
print("\n--- Stage D: Estimating the model's performance ---")
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nFinal Test accuracy: {test_acc * 100:.2f}%")

print("\nMaking a prediction on a single test image...")
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# (32, 32, 3) -> (1, 32, 32, 3) to make a batch of 1
img = np.expand_dims(x_test[0], axis=0)
prediction = model.predict(img)

predicted_class = np.argmax(prediction)
true_class = np.argmax(y_test[0])

print(f"The model predicts the image is a '{class_names[predicted_class]}'.")
print(f"The true class label is a '{class_names[true_class]}'.")

--- Stage A: Loading and preparing the CIFAR-10 dataset ---


 35332096/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:10:20 31us/step